# NIDS-VAE: Walkthrough đầy đủ từ CSV đến dự đoán

Notebook này giải thích toàn bộ dự án NIDS-VAE bằng tiếng Việt, dành cho người học cần hiểu, demo và bảo vệ đồ án. Nội dung bám vào code, docs và artifacts hiện có trong repository:

- `artifacts/feature_schema/feature_columns.json`
- `artifacts/scaler/imputation_medians.json`
- `artifacts/scaler/scaler.joblib`
- `artifacts/models/model_config.json`
- `artifacts/models/vae_best.pth`
- `artifacts/models/training_history.json`
- `artifacts/models/training_summary.json`
- `artifacts/threshold/threshold.json`
- `artifacts/sample_batch/fixed_batch.csv`
- code backend trong `backend/app/core/` và `backend/app/models/`

Notebook không retrain model, không tải lại CICIDS2017 và không chỉnh sửa artifacts.

**Cách chạy:** mở notebook từ thư mục gốc repository `nids-vae-project/` để các đường dẫn tương đối hoạt động đúng.

## 1. Mục tiêu bài toán

Dự án xây dựng một **Network Intrusion Detection System** dùng **Variational Autoencoder (VAE)** để phát hiện luồng mạng bất thường.

| Thành phần | Ý nghĩa |
|---|---|
| Input | Dữ liệu network flow dạng CSV, theo kiểu CICIDS2017 |
| Output | `normal` hoặc `anomaly` cho từng flow |
| Model | Variational Autoencoder |
| Dataset | CICIDS2017 |
| Ý tưởng cốt lõi | Train VAE chỉ bằng flow `BENIGN`; flow nào khó reconstruct sẽ có reconstruction error cao và bị xem là bất thường |

Pipeline tổng quát:

```text
CSV/PCAP -> Trích xuất đặc trưng -> Chọn 66 feature -> Điền missing values -> Chuẩn hóa -> VAE -> Reconstruction Error -> Threshold -> Normal / Anomaly -> Dashboard
```

Điểm cần nhớ khi bảo vệ:

- VAE không học trực tiếp nhãn attack.
- Model học “hình dạng” của traffic bình thường.
- Điểm anomaly là **reconstruction error**, tức độ lệch giữa input đã chuẩn hóa và output tái tạo của VAE.
- Threshold là nguồn quyết định cuối cùng: `error > threshold` nghĩa là anomaly.

In [1]:
from pathlib import Path
import json
import re
import sys

import numpy as np
import pandas as pd

# Xác định thư mục gốc dự án. Notebook nên chạy từ repo root,
# nhưng fallback này giúp nếu bạn lỡ mở từ thư mục notebooks/.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "CLAUDE.md").exists() and (PROJECT_ROOT.parent / "CLAUDE.md").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "CLAUDE.md").exists():
    raise RuntimeError(
        "Không tìm thấy CLAUDE.md. Hãy chạy notebook từ thư mục gốc nids-vae-project/."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

FEATURE_SCHEMA_PATH = ARTIFACTS_DIR / "feature_schema" / "feature_columns.json"

IMPUTATION_MEDIANS_PATH = ARTIFACTS_DIR / "scaler" / "imputation_medians.json"
SCALER_PATH = ARTIFACTS_DIR / "scaler" / "scaler.joblib"

MODEL_CONFIG_PATH = ARTIFACTS_DIR / "models" / "model_config.json"
MODEL_CHECKPOINT_PATH = ARTIFACTS_DIR / "models" / "vae_best.pth"
TRAINING_HISTORY_PATH = ARTIFACTS_DIR / "models" / "training_history.json"
TRAINING_SUMMARY_PATH = ARTIFACTS_DIR / "models" / "training_summary.json"

THRESHOLD_PATH = ARTIFACTS_DIR / "threshold" / "threshold.json"

SAMPLE_BATCH_PATH = ARTIFACTS_DIR / "sample_batch" / "fixed_batch.csv"

# =========================
# Final experiment/report artifacts
# Dùng cho notebook bảo vệ/báo cáo để lấy Best Epoch = 58, Final Epoch = 68.
# =========================

BEST_EPOCH_DIR = ARTIFACTS_DIR / "experiments" / "best_epoch"
BEST_EPOCH_SUMMARY_PATH = BEST_EPOCH_DIR / "best_epoch_summary.csv"
BEST_VS_FINAL_EPOCH_PATH = BEST_EPOCH_DIR / "best_vs_final_epoch.csv"
BEST_EPOCH_REPORT_PATH = BEST_EPOCH_DIR / "best_epoch_report.txt"

THRESHOLD_EXPERIMENT_DIR = ARTIFACTS_DIR / "experiments" / "threshold"
THRESHOLD_VALUES_PATH = THRESHOLD_EXPERIMENT_DIR / "threshold_values.csv"


def resolve_path(path: str | Path) -> Path:
    '''Chuyển path tương đối về path tuyệt đối tính từ repo root.'''
    path = Path(path)
    return path if path.is_absolute() else PROJECT_ROOT / path


def require_file(path: str | Path, label: str = "file") -> Path:
    '''Kiểm tra file bắt buộc có tồn tại hay không.'''
    path = resolve_path(path)
    if not path.exists():
        raise FileNotFoundError(f"Không tìm thấy {label}: {path}")
    return path


def load_json(path: str | Path) -> dict | list:
    '''Đọc JSON chuẩn và báo lỗi rõ nếu file chứa Git conflict markers.'''
    path = require_file(path, "JSON artifact")
    text = path.read_text(encoding="utf-8")

    # Không âm thầm bỏ qua conflict marker vì backend cũng sẽ không đọc được JSON này.
    if any(marker in text for marker in ("<<<<<<<", "=======", ">>>>>>>")):
        raise ValueError(
            f"{path.relative_to(PROJECT_ROOT)} đang chứa Git conflict markers; "
            "hãy resolve file này trước khi dùng backend production."
        )

    return json.loads(text)


def load_feature_columns() -> list[str]:
    '''Tải danh sách 66 feature đúng thứ tự huấn luyện.'''
    data = load_json(FEATURE_SCHEMA_PATH)
    if isinstance(data, dict):
        columns = data.get("feature_columns")
    else:
        columns = data
    if not columns:
        raise ValueError("feature_columns.json không có danh sách feature hợp lệ.")
    return list(columns)


def load_imputation_medians() -> dict[str, float]:
    '''Tải median dùng để điền NaN trong inference.'''
    return dict(load_json(IMPUTATION_MEDIANS_PATH))


def load_scaler():
    '''Tải StandardScaler đã fit trên train set, không fit lại.'''
    import joblib

    return joblib.load(require_file(SCALER_PATH, "scaler.joblib"))


def load_training_history_dataframe(path: str | Path = TRAINING_HISTORY_PATH) -> pd.DataFrame:
    '''Tạo DataFrame lịch sử train từ đúng cấu trúc JSON hiện có.'''
    data = load_json(path)

    # training_history.json hiện là dict các list loss theo epoch.
    # Hàm vẫn hỗ trợ list[dict] để notebook không vỡ nếu artifact sau này đổi format có kiểm soát.
    if isinstance(data, list):
        history_df = pd.DataFrame(data)
    elif isinstance(data, dict):
        list_values = {key: value for key, value in data.items() if isinstance(value, list)}
        if not list_values:
            raise ValueError("training_history.json không có list metric theo epoch.")

        lengths = {key: len(value) for key, value in list_values.items()}
        if len(set(lengths.values())) != 1:
            raise ValueError(f"Các list trong training_history.json có độ dài không đều: {lengths}")

        history_df = pd.DataFrame(list_values)
    else:
        raise TypeError("training_history.json phải là dict các list hoặc list các record.")

    if "epoch" not in history_df.columns:
        # Epoch trong artifact được lưu implicit theo vị trí list, bắt đầu từ 1.
        history_df.insert(0, "epoch", np.arange(1, len(history_df) + 1, dtype=int))

    rename_map = {
        "loss": "train_loss",
        "total_loss": "train_loss",
        "valid_loss": "val_loss",
        "validation_loss": "val_loss",
    }
    history_df = history_df.rename(columns={k: v for k, v in rename_map.items() if k in history_df.columns})

    required_cols = ["epoch", "train_loss", "val_loss"]
    missing = [col for col in required_cols if col not in history_df.columns]
    if missing:
        raise ValueError(f"history_df thiếu cột bắt buộc: {missing}")

    return history_df


def load_best_epoch_metrics(path: str | Path = BEST_EPOCH_SUMMARY_PATH) -> tuple[pd.DataFrame, pd.Series]:
    '''Đọc best_epoch_summary.csv và trả về bảng cùng Series metric->value.'''
    path = require_file(path, "best_epoch_summary.csv")
    summary_df = pd.read_csv(path)

    required_cols = {"metric", "value"}
    if not required_cols.issubset(summary_df.columns):
        raise ValueError(f"best_epoch_summary.csv phải có cột {sorted(required_cols)}")

    metrics = summary_df.set_index("metric")["value"].astype(float)
    required_metrics = [
        "best_epoch",
        "final_epoch",
        "best_train_loss",
        "best_val_loss",
        "final_train_loss",
        "final_val_loss",
    ]
    missing = [name for name in required_metrics if name not in metrics.index]
    if missing:
        raise ValueError(f"best_epoch_summary.csv thiếu metric: {missing}")

    return summary_df, metrics


def validate_and_align_schema(
    df: pd.DataFrame,
    feature_columns: list[str],
) -> tuple[pd.DataFrame, list[str]]:
    '''Kiểm tra đủ cột bắt buộc và sắp xếp đúng thứ tự feature schema.'''
    df = df.copy()

    # Chỉ loại bỏ khoảng trắng đầu/cuối tên cột.
    # Không lower-case vì feature_columns.json giữ tên cột gốc khi train.
    df.columns = df.columns.str.strip()

    missing_cols = [col for col in feature_columns if col not in df.columns]
    extra_cols = [col for col in df.columns if col not in feature_columns]

    if missing_cols:
        raise ValueError(
            f"CSV thiếu {len(missing_cols)} cột bắt buộc: {missing_cols}"
        )

    aligned_df = df[feature_columns].copy()
    return aligned_df, extra_cols


def clean_and_impute(df: pd.DataFrame, medians: dict[str, float]) -> pd.DataFrame:
    '''Ép numeric, xử lý NaN/Infinity và điền thiếu bằng median train set.'''
    df = df.copy()

    # Ép kiểu numeric: giá trị không phải số sẽ thành NaN để impute.
    df = df.apply(pd.to_numeric, errors="coerce")

    # Infinity không hợp lệ với scaler/model nên chuyển thành NaN.
    df = df.replace([np.inf, -np.inf], np.nan)

    missing_median_cols = [col for col in df.columns if col not in medians]
    if missing_median_cols:
        raise ValueError(
            f"Thiếu median cho {len(missing_median_cols)} cột: {missing_median_cols}"
        )

    # Median chỉ dùng để lấp giá trị thiếu bên trong các cột đã tồn tại.
    # Không dùng median để tạo cột bắt buộc bị thiếu.
    df = df.fillna(pd.Series(medians))

    if df.isna().any().any():
        missing = df.columns[df.isna().any()].tolist()
        raise ValueError(f"Vẫn còn NaN sau imputation ở các cột: {missing}")
    if np.isinf(df.to_numpy()).any():
        raise ValueError("Vẫn còn Infinity sau khi làm sạch dữ liệu.")

    return df


def preprocess(file_path: str | Path) -> np.ndarray:
    '''Pipeline inference đầy đủ: CSV -> schema -> clean -> scaler.transform().'''
    df = pd.read_csv(require_file(file_path, "CSV input"))

    # Chỉ loại bỏ khoảng trắng đầu/cuối tên cột.
    df.columns = df.columns.str.strip()

    feature_columns = load_feature_columns()
    medians = load_imputation_medians()
    scaler = load_scaler()

    # Kiểm tra schema: thiếu cột bắt buộc là lỗi; cột thừa được bỏ qua.
    df, _extra_cols = validate_and_align_schema(df, feature_columns)

    # Ép kiểu numeric, thay Infinity bằng NaN, rồi điền median train set.
    df = clean_and_impute(df, medians)

    # Chuẩn hóa bằng scaler đã train. Tuyệt đối không fit lại scaler ở inference.
    X_scaled = scaler.transform(df)
    return X_scaled


def inspect_state_dict(path: str | Path):
    '''Tải PyTorch state_dict và trả về bảng key/shape để inspect weights.'''
    import torch

    path = require_file(path, "model checkpoint")
    try:
        state_dict = torch.load(path, map_location="cpu", weights_only=True)
    except TypeError:
        # Tương thích với phiên bản torch cũ chưa có weights_only.
        state_dict = torch.load(path, map_location="cpu")

    if isinstance(state_dict, dict) and "state_dict" in state_dict:
        state_dict = state_dict["state_dict"]

    rows = []
    for key, value in state_dict.items():
        shape = tuple(value.shape) if hasattr(value, "shape") else None
        rows.append({"key": key, "shape": shape, "dtype": str(getattr(value, "dtype", ""))})

    return state_dict, pd.DataFrame(rows)


print(f"Repo root: {PROJECT_ROOT}")
print("Đã khai báo helper functions và artifact paths.")


Repo root: D:\nids-vae-project
Đã khai báo helper functions và artifact paths.


## 2. Dữ liệu đầu vào ban đầu

CICIDS2017 ở dạng flow-level CSV: mỗi dòng là một network flow đã được trích xuất đặc trưng, không phải từng packet riêng lẻ. Một flow thường mô tả giao tiếp giữa hai endpoint trong một khoảng thời gian.

Ví dụ các cột thường gặp:

| Cột | Ý nghĩa trực giác |
|---|---|
| `Destination Port` | Port đích, ví dụ 80/443/22 |
| `Flow Duration` | Thời lượng flow |
| `Total Fwd Packets` | Số packet chiều forward |
| `Flow Bytes/s` | Tốc độ byte/giây |
| `Packet Length Mean` | Độ dài packet trung bình |
| `Label` | Nhãn trong raw data, nếu còn tồn tại; pipeline model sẽ bỏ nhãn khỏi feature |

Trong repository này, file mẫu đã được lưu ở `artifacts/sample_batch/fixed_batch.csv`. Đây là batch 128 flow unscaled để test inference.

In [2]:
# Đọc sample CSV có sẵn trong artifacts.
sample_df = pd.read_csv(require_file(SAMPLE_BATCH_PATH, "fixed_batch.csv"))
sample_df.columns = [str(col).strip() for col in sample_df.columns]

print("Kích thước sample:", sample_df.shape)
print("10 cột đầu tiên:")
print(sample_df.columns[:10].tolist())

# Hiển thị một vài dòng đầu để thấy dữ liệu flow-level trông như thế nào.
sample_df.head(5)

Kích thước sample: (128, 66)
10 cột đầu tiên:
['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std']


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min
0,54865,3,2,0,12,0,6,6,6.0,0.0,...,1,20,0.0,0.0,0,0,0.0,0.0,0,0
1,55054,109,1,1,6,6,6,6,6.0,0.0,...,0,20,0.0,0.0,0,0,0.0,0.0,0,0
2,55055,52,1,1,6,6,6,6,6.0,0.0,...,0,20,0.0,0.0,0,0,0.0,0.0,0,0
3,46236,34,1,1,6,6,6,6,6.0,0.0,...,0,20,0.0,0.0,0,0,0.0,0.0,0,0
4,54863,3,2,0,12,0,6,6,6.0,0.0,...,1,20,0.0,0.0,0,0,0.0,0.0,0,0


## 3. Tiền xử lý dữ liệu

Pipeline tiền xử lý trong notebook này bám vào artifact runtime và contract inference hiện tại:

```text
CSV
↓
Validate required schema
↓
Keep exactly 66 features
↓
Reorder columns according to feature_columns.json
↓
Convert to numeric
↓
Replace ±Inf with NaN
↓
Fill NaN using imputation_medians.json
↓
Apply scaler.joblib
↓
VAE
```

Điểm quan trọng:

- **Missing required columns → ERROR.** CSV phải có đủ 66 cột bắt buộc theo `feature_columns.json`.
- **NaN values inside existing columns → Fill using median.** Median lấy từ `imputation_medians.json`, đã tính từ train set.
- Median **không** được dùng để tạo các cột bắt buộc bị thiếu.
- Feature order là một phần của model contract. Nếu cùng 66 cột nhưng đảo thứ tự, input vào VAE sẽ sai nghĩa.


In [3]:
# Tải feature schema từ artifact.
feature_columns = load_feature_columns()

print("Số feature:", len(feature_columns))
print("10 feature đầu tiên:")
for i, name in enumerate(feature_columns[:10], start=1):
    print(f"{i:02d}. {name}")

Số feature: 66
10 feature đầu tiên:
01. Destination Port
02. Flow Duration
03. Total Fwd Packets
04. Total Backward Packets
05. Total Length of Fwd Packets
06. Total Length of Bwd Packets
07. Fwd Packet Length Max
08. Fwd Packet Length Min
09. Fwd Packet Length Mean
10. Fwd Packet Length Std


In [4]:
# Tải median imputation từ artifact.
medians = load_imputation_medians()

median_examples = [
    "Destination Port",
    "Flow Duration",
    "Flow Bytes/s",
    "Flow Packets/s",
    "Packet Length Mean",
]

pd.DataFrame(
    [{"feature": name, "median_from_train": medians.get(name)} for name in median_examples]
)

,feature,median_from_train
0,Destination Port,80.000000
1,Flow Duration,36816.000000
2,Flow Bytes/s,4969.431458
3,Flow Packets/s,88.368497
4,Packet Length Mean,61.600000


In [5]:
# Kiểm tra sample CSV có đủ các cột bắt buộc không.
missing_required = [col for col in feature_columns if col not in sample_df.columns]
extra_columns = [col for col in sample_df.columns if col not in feature_columns]

print(f"Số cột bắt buộc bị thiếu: {len(missing_required)}")
print(f"Số cột thừa ngoài schema: {len(extra_columns)}")

if missing_required:
    print("Một số cột thiếu:", missing_required[:10])
if extra_columns:
    print("Một số cột thừa:", extra_columns[:10])

# Align về đúng 66 feature theo thứ tự schema.
aligned_df, extra_cols = validate_and_align_schema(
    sample_df,
    feature_columns
)

clean_df = clean_and_impute(
    aligned_df,
    medians
)

print("Shape sau align + impute:", clean_df.shape)
print("Còn NaN:", int(clean_df.isna().sum().sum()))
print("Còn Inf:", int(np.isinf(clean_df.to_numpy()).sum()))

clean_df.head(3)

Số cột bắt buộc bị thiếu: 0
Số cột thừa ngoài schema: 0
Shape sau align + impute: (128, 66)
Còn NaN: 0
Còn Inf: 0


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min
0,54865,3,2,0,12,0,6,6,6.0,0.0,...,1,20,0.0,0.0,0,0,0.0,0.0,0,0
1,55054,109,1,1,6,6,6,6,6.0,0.0,...,0,20,0.0,0.0,0,0,0.0,0.0,0,0
2,55055,52,1,1,6,6,6,6,6.0,0.0,...,0,20,0.0,0.0,0,0,0.0,0.0,0,0


## 4. Chuẩn hóa dữ liệu bằng `scaler.joblib`

Project dùng `StandardScaler`, với công thức:

```text
x_scaled = (x - mean) / std
```

Trong training:

- `scaler.fit(...)` chỉ chạy trên train set BENIGN.
- Train/validation/test được transform bằng scaler đó.

Trong prediction/backend:

- Chỉ dùng `scaler.transform(...)`.
- Tuyệt đối không dùng `fit_transform(...)`, vì fit lại scaler trên file user upload sẽ làm training và deployment lệch nhau.

In [6]:
scaler = load_scaler()

print("Scaler type:", type(scaler))
print("mean_.shape :", scaler.mean_.shape)
print("scale_.shape:", scaler.scale_.shape)

scaler_rows = []
for name in ["Destination Port", "Flow Duration", "Flow Bytes/s", "Flow Packets/s", "Packet Length Mean"]:
    idx = feature_columns.index(name)
    scaler_rows.append(
        {
            "feature": name,
            "mean": scaler.mean_[idx],
            "std": scaler.scale_[idx],
        }
    )

pd.DataFrame(scaler_rows)

Scaler type: <class 'sklearn.preprocessing._data.StandardScaler'>
mean_.shape : (66,)
scale_.shape: (66,)


,feature,mean,std
0,Destination Port,1.117736e+04,2.179576e+04
1,Flow Duration,1.093916e+07,2.941495e+07
2,Flow Bytes/s,1.483113e+06,2.506125e+07
3,Flow Packets/s,5.667531e+04,2.163655e+05
4,Packet Length Mean,1.060934e+02,1.632021e+02


In [7]:
# Ví dụ số học thật với feature Flow Duration.
feature_name = "Flow Duration"
idx = feature_columns.index(feature_name)

raw_value = float(clean_df.iloc[0][feature_name])
mean_value = float(scaler.mean_[idx])
std_value = float(scaler.scale_[idx])
scaled_manual = (raw_value - mean_value) / std_value
scaled_by_scaler = float(scaler.transform(clean_df.iloc[[0]])[0, idx])

pd.DataFrame(
    [
        {
            "feature": feature_name,
            "raw": raw_value,
            "mean": mean_value,
            "std": std_value,
            "scaled_manual": scaled_manual,
            "scaled_by_scaler": scaled_by_scaler,
        }
    ]
)

,feature,raw,mean,std,scaled_manual,scaled_by_scaler
0,Flow Duration,3.0,1.093916e+07,2.941495e+07,-0.371891,-0.371891


In [8]:
# So sánh trước/sau khi scale cho một flow đầu tiên.
X_scaled_one = scaler.transform(clean_df.iloc[[0]])

before_after = pd.DataFrame(
    {
        "feature": feature_columns[:10],
        "raw_or_imputed": clean_df.iloc[0][feature_columns[:10]].to_numpy(),
        "scaled": X_scaled_one[0, :10],
    }
)

before_after

,feature,raw_or_imputed,scaled
0,Destination Port,54865.0,2.004410
1,Flow Duration,3.0,-0.371891
2,Total Fwd Packets,2.0,-0.009626
3,Total Backward Packets,0.0,-0.009975
4,Total Length of Fwd Packets,12.0,-0.078683
5,Total Length of Bwd Packets,0.0,-0.007149
6,Fwd Packet Length Max,6.0,-0.421792
7,Fwd Packet Length Min,6.0,-0.378325
8,Fwd Packet Length Mean,6.0,-0.489450
9,Fwd Packet Length Std,0.0,-0.403630


## 5. Chia dữ liệu train / validation / test

Theo docs và `scripts/clean_data.py`, project chia dữ liệu như sau:

| Split | Nguồn | Nhãn |
|---|---|---|
| Train | 80% Monday | BENIGN only |
| Validation | 20% Monday | BENIGN only |
| Test | Tuesday-Friday / non-Monday | BENIGN + ATTACK |

Số lượng đã xác nhận trong artifacts/docs:

| Split | Shape | Ghi chú |
|---|---:|---|
| Train | 402,229 x 66 | toàn BENIGN |
| Validation | 100,558 x 66 | toàn BENIGN |
| Test | 2,019,575 x 66 | BENIGN + Attack |

VAE train chỉ trên BENIGN để học “profile bình thường”. Khi gặp flow attack hoặc flow lạ, model thường tái tạo kém hơn, làm reconstruction error tăng.

In [9]:
# ==========================================================
# Đọc kết quả Best Epoch cuối cùng dùng trong báo cáo
# Nguồn sự thật: artifacts/experiments/best_epoch/best_epoch_summary.csv
# ==========================================================

best_epoch_summary = pd.read_csv(BEST_EPOCH_SUMMARY_PATH)
best_epoch_summary, best_epoch_metrics = load_best_epoch_metrics(BEST_EPOCH_SUMMARY_PATH)
training_summary = load_json(TRAINING_SUMMARY_PATH)
history_df = load_training_history_dataframe(TRAINING_HISTORY_PATH)

# ==========================================================
# Trích xuất các chỉ số quan trọng từ CSV báo cáo cuối cùng
# ==========================================================

best_epoch = int(best_epoch_metrics["best_epoch"])
final_epoch = int(best_epoch_metrics["final_epoch"])
best_train_loss = float(best_epoch_metrics["best_train_loss"])
best_val_loss = float(best_epoch_metrics["best_val_loss"])
final_train_loss = float(best_epoch_metrics["final_train_loss"])
final_val_loss = float(best_epoch_metrics["final_val_loss"])

# ==========================================================
# Hiển thị kết quả để dùng trong phần bảo vệ đồ án
# ==========================================================

display(best_epoch_summary)

print("===== KẾT QUẢ HUẤN LUYỆN CUỐI CÙNG =====")
print(f"Best Epoch      : {best_epoch}")
print(f"Final Epoch     : {final_epoch}")
print(f"Best Train Loss : {best_train_loss:.6f}")
print(f"Best Val Loss   : {best_val_loss:.6f}")
print(f"Final Train Loss: {final_train_loss:.6f}")
print(f"Final Val Loss  : {final_val_loss:.6f}")

print("\n===== TRAINING HISTORY ARTIFACT =====")
print("Đã tạo history_df từ cấu trúc thật của training_history.json.")
print("Các cột chính dùng cho learning curve:", ["epoch", "train_loss", "val_loss"])
print(
    "Lưu ý: Best/Final Epoch báo cáo luôn lấy từ best_epoch_summary.csv "
    "khi artifact learning-curve không cùng phạm vi epoch."
)

history_df[["epoch", "train_loss", "val_loss"]].head(3)


,metric,value
0,best_epoch,58.000000
1,best_val_loss,0.740856
2,best_train_loss,0.772250
3,final_epoch,68.000000
4,final_val_loss,0.746338
5,final_train_loss,0.765520
6,delta_val_loss_final_minus_best,0.005482
7,generalization_gap_best,-0.031394
8,generalization_gap_final,-0.019182


===== KẾT QUẢ HUẤN LUYỆN CUỐI CÙNG =====
Best Epoch      : 58
Final Epoch     : 68
Best Train Loss : 0.772250
Best Val Loss   : 0.740856
Final Train Loss: 0.765520
Final Val Loss  : 0.746338

===== TRAINING HISTORY ARTIFACT =====
Đã tạo history_df từ cấu trúc thật của training_history.json.
Các cột chính dùng cho learning curve: ['epoch', 'train_loss', 'val_loss']
Lưu ý: Best/Final Epoch báo cáo luôn lấy từ best_epoch_summary.csv khi artifact learning-curve không cùng phạm vi epoch.


,epoch,train_loss,val_loss
0,1,1242.621565,0.376873
1,2,0.640047,0.392957
2,3,1.339212,0.405747


## 6. Huấn luyện mô hình VAE: giải thích chi tiết

Đây là phần quan trọng nhất khi bảo vệ. Kiến trúc thật được lấy từ `artifacts/models/model_config.json`.

In [10]:
model_config = load_json(MODEL_CONFIG_PATH)
model_config

{'architecture': 'VAE',
 'input_dim': 66,
 'latent_dim': 16,
 'hidden_dims': [128, 64],
 'beta': 1.0,
 'activation': 'ReLU',
 'output_activation': 'Linear'}

Với config hiện tại, kiến trúc là:

```text
66 features
-> Encoder: 66 -> 128 -> 64
-> mu và logvar: 64 -> latent_dim
-> z = mu + sigma * epsilon
-> Decoder: latent_dim -> 64 -> 128 -> 66
-> x_hat
```

Nếu `latent_dim` trong config khác 16, hãy dùng đúng giá trị trong cell phía trên. Trong artifact hiện tại, `latent_dim = 16`.

### 6.1 Encoder layer 1: 66 -> 128

Mỗi neuron hidden ở layer đầu nhận toàn bộ 66 feature đầu vào đã chuẩn hóa.

```text
x1  ─┐
x2  ─┤
x3  ─┤-> h1 cần 66 weight + 1 bias
... ─┤
x66 ─┘
```

Công thức của một neuron:

```text
h1 = w1*x1 + w2*x2 + ... + w66*x66 + b1
```

Trong đó:

- `x1...x66` là giá trị feature đã chuẩn hóa.
- `w1...w66` là trọng số học được.
- `b1` là bias học được.
- 128 neuron nghĩa là có 128 “góc nhìn có trọng số” khác nhau trên cùng 66 feature.
- Layer `66 -> 128` có `66 x 128 = 8,448` weights và `128` biases.

In [11]:
state_dict, state_shapes = inspect_state_dict(MODEL_CHECKPOINT_PATH)
state_shapes

,key,shape,dtype
0,encoder.0.weight,"(128, 66)",torch.float32
1,encoder.0.bias,"(128,)",torch.float32
2,encoder.2.weight,"(64, 128)",torch.float32
3,encoder.2.bias,"(64,)",torch.float32
4,fc_mu.weight,"(16, 64)",torch.float32
5,fc_mu.bias,"(16,)",torch.float32
6,fc_logvar.weight,"(16, 64)",torch.float32
7,fc_logvar.bias,"(16,)",torch.float32
8,decoder.0.weight,"(64, 16)",torch.float32
9,decoder.0.bias,"(64,)",torch.float32


In [12]:
def find_tensor_key(
    state_dict: dict,
    *,
    ndim: int | None = None,
    shape: tuple[int, ...] | None = None,
    keywords: tuple[str, ...] = (),
) -> str | None:
    '''Tìm key tensor theo shape và keyword để không phụ thuộc cứng vào tên layer.'''
    for key, value in state_dict.items():
        if not hasattr(value, "shape"):
            continue
        if ndim is not None and value.ndim != ndim:
            continue
        if shape is not None and tuple(value.shape) != shape:
            continue
        if keywords and not all(word.lower() in key.lower() for word in keywords):
            continue
        return key
    return None


input_dim = int(model_config["input_dim"])
hidden_dims = list(model_config["hidden_dims"])
latent_dim = int(model_config["latent_dim"])

first_weight_key = find_tensor_key(
    state_dict,
    ndim=2,
    shape=(hidden_dims[0], input_dim),
    keywords=("encoder",),
)
first_bias_key = find_tensor_key(
    state_dict,
    ndim=1,
    shape=(hidden_dims[0],),
    keywords=("encoder",),
)

print("Encoder layer 1 weight key:", first_weight_key)
print("Encoder layer 1 bias key  :", first_bias_key)
print("Weight shape:", tuple(state_dict[first_weight_key].shape))
print("Bias shape  :", tuple(state_dict[first_bias_key].shape))

# Neuron h1 là hàng đầu tiên của ma trận weight.
h1_weights = state_dict[first_weight_key][0].detach().cpu().numpy()
h1_bias = float(state_dict[first_bias_key][0].detach().cpu())

print("10 weight đầu tiên của neuron h1:")
print(h1_weights[:10])
print("Bias của neuron h1:", h1_bias)

Encoder layer 1 weight key: encoder.0.weight
Encoder layer 1 bias key  : encoder.0.bias
Weight shape: (128, 66)
Bias shape  : (128,)
10 weight đầu tiên của neuron h1:
[-0.0639949   0.03539158 -0.01339807  0.01900164 -0.07548156 -0.00053713
 -0.00890502 -0.05692722 -0.00488241  0.02298037]
Bias của neuron h1: -0.6468222141265869


### 6.2 Encoder layer 2: 128 -> 64

Layer thứ hai có ma trận weight riêng, được khởi tạo riêng và học riêng.

- `W1` cho `66 -> 128` khác với `W2` cho `128 -> 64`.
- `W1` thường có shape `[128, 66]`.
- `W2` thường có shape `[64, 128]`.
- Batch 2 không random lại weight.
- Batch 2 dùng weight đã được cập nhật sau Batch 1, rồi cập nhật tiếp.

Timeline huấn luyện:

```text
Model initialization -> random W một lần
Batch 1 -> update W
Batch 2 -> dùng W đã update -> update tiếp
...
Epoch 2 -> tiếp tục từ Epoch 1, không reset
```

In [13]:
second_weight_key = find_tensor_key(
    state_dict,
    ndim=2,
    shape=(hidden_dims[1], hidden_dims[0]),
    keywords=("encoder",),
)
second_bias_key = find_tensor_key(
    state_dict,
    ndim=1,
    shape=(hidden_dims[1],),
    keywords=("encoder",),
)

pd.DataFrame(
    [
        {"layer": "encoder 66->128", "weight_key": first_weight_key, "shape": tuple(state_dict[first_weight_key].shape)},
        {"layer": "encoder 128->64", "weight_key": second_weight_key, "shape": tuple(state_dict[second_weight_key].shape)},
        {"layer": "encoder 128->64 bias", "weight_key": second_bias_key, "shape": tuple(state_dict[second_bias_key].shape)},
    ]
)

,layer,weight_key,shape
0,encoder 66->128,encoder.0.weight,"(128, 66)"
1,encoder 128->64,encoder.2.weight,"(64, 128)"
2,encoder 128->64 bias,encoder.2.bias,"(64,)"


### 6.3 `mu` và `logvar`

VAE không xuất trực tiếp một latent vector cố định. Sau encoder, model sinh ra hai vector:

- `mu`: tâm của phân phối latent.
- `logvar`: log variance của phân phối latent.

Công thức tuyến tính:

```text
mu = W_mu * h + b_mu
logvar = W_logvar * h + b_logvar
```

Hai nhánh này cho phép model biểu diễn mỗi input thành một phân phối xác suất trong latent space.

In [14]:
mu_weight_key = find_tensor_key(state_dict, ndim=2, shape=(latent_dim, hidden_dims[-1]), keywords=("mu",))
mu_bias_key = find_tensor_key(state_dict, ndim=1, shape=(latent_dim,), keywords=("mu",))
logvar_weight_key = find_tensor_key(state_dict, ndim=2, shape=(latent_dim, hidden_dims[-1]), keywords=("logvar",))
logvar_bias_key = find_tensor_key(state_dict, ndim=1, shape=(latent_dim,), keywords=("logvar",))

rows = []
for name, key in [
    ("fc_mu.weight", mu_weight_key),
    ("fc_mu.bias", mu_bias_key),
    ("fc_logvar.weight", logvar_weight_key),
    ("fc_logvar.bias", logvar_bias_key),
]:
    rows.append({"name": name, "key": key, "shape": tuple(state_dict[key].shape)})

display(pd.DataFrame(rows))

print("5 weight đầu tiên của latent dim 0 trong fc_mu:")
print(state_dict[mu_weight_key][0, :5].detach().cpu().numpy())
print("Bias latent dim 0 của fc_mu:", float(state_dict[mu_bias_key][0]))

print("5 weight đầu tiên của latent dim 0 trong fc_logvar:")
print(state_dict[logvar_weight_key][0, :5].detach().cpu().numpy())
print("Bias latent dim 0 của fc_logvar:", float(state_dict[logvar_bias_key][0]))

,name,key,shape
0,fc_mu.weight,fc_mu.weight,"(16, 64)"
1,fc_mu.bias,fc_mu.bias,"(16,)"
2,fc_logvar.weight,fc_logvar.weight,"(16, 64)"
3,fc_logvar.bias,fc_logvar.bias,"(16,)"


5 weight đầu tiên của latent dim 0 trong fc_mu:
[-2.5575080e-09  3.3011228e-02 -9.3027633e-03  1.3269533e-02
 -3.8036269e-03]
Bias latent dim 0 của fc_mu: -0.004593577701598406
5 weight đầu tiên của latent dim 0 trong fc_logvar:
[-2.1579446e-38  2.2656195e-02 -8.8059101e-03  2.7233882e-02
  6.0783625e-03]
Bias latent dim 0 của fc_logvar: -0.0010013034334406257


### 6.4 Reparameterization Trick

Công thức đúng:

```text
sigma = exp(0.5 * logvar)
epsilon ~ N(0, 1)
z = mu + sigma * epsilon
```

Không nhân `epsilon` trực tiếp với `logvar`. `logvar` phải được chuyển thành độ lệch chuẩn `sigma`.

Ví dụ số:

```text
mu = 0.21
logvar = 0.11
sigma = exp(0.5 * 0.11)
epsilon = 0.5
z = mu + sigma * epsilon
```

`epsilon` là nhiễu ngẫu nhiên từ phân phối chuẩn. Cách viết này giúp model vẫn sampling được nhưng gradient vẫn truyền ngược qua `mu` và `logvar`.

In [15]:
import torch

# Ví dụ số học cố định.
mu_example = torch.tensor([[0.21]])
logvar_example = torch.tensor([[0.11]])
sigma_example = torch.exp(0.5 * logvar_example)
epsilon_fixed = torch.tensor([[0.5]])
z_example = mu_example + sigma_example * epsilon_fixed

print("sigma =", float(sigma_example))
print("z     =", float(z_example))

# Ví dụ giống code thật: epsilon được sinh bằng torch.randn_like.
mu_vec = torch.zeros((2, latent_dim))
logvar_vec = torch.zeros((2, latent_dim))
std_vec = torch.exp(0.5 * logvar_vec)
epsilon = torch.randn_like(std_vec)
z_vec = mu_vec + std_vec * epsilon

print("Shape epsilon:", tuple(epsilon.shape))
print("Shape z      :", tuple(z_vec.shape))

sigma = 1.056540608406067
z     = 0.7382702827453613
Shape epsilon: (2, 16)
Shape z      : (2, 16)


### 6.5 Decoder và reconstruction

Decoder nhận latent vector `z` và cố gắng tái tạo lại input:

```text
z -> Decoder -> x_hat
```

Ví dụ trực giác:

```text
x     = [0.50, 1.20, 0.80, ...]
x_hat = [0.48, 1.18, 0.75, ...]
```

Nếu flow giống normal traffic, `x_hat` thường gần `x`. Nếu flow lạ hoặc attack, reconstruction có xu hướng kém hơn.

In [16]:
decoder_first_weight_key = find_tensor_key(
    state_dict,
    ndim=2,
    shape=(hidden_dims[-1], latent_dim),
    keywords=("decoder",),
)
decoder_output_weight_key = find_tensor_key(
    state_dict,
    ndim=2,
    shape=(input_dim, hidden_dims[0]),
    keywords=("decoder",),
)

pd.DataFrame(
    [
        {"part": "decoder first layer", "key": decoder_first_weight_key, "shape": tuple(state_dict[decoder_first_weight_key].shape)},
        {"part": "decoder output layer", "key": decoder_output_weight_key, "shape": tuple(state_dict[decoder_output_weight_key].shape)},
    ]
)

,part,key,shape
0,decoder first layer,decoder.0.weight,"(64, 16)"
1,decoder output layer,decoder.4.weight,"(66, 128)"


### 6.6 Loss function

VAE loss gồm hai phần:

```text
Total Loss = Reconstruction Loss + KL Divergence
```

Reconstruction loss đo `x` và `x_hat` khác nhau bao nhiêu:

```text
Recon Loss = MSE(x, x_hat)
```

KL Divergence kéo latent distribution về gần chuẩn `N(0,1)`:

```text
KL = -0.5 * sum(1 + logvar - mu^2 - exp(logvar))
```

Trong project, `scripts/train.py` dùng reconstruction MSE + beta * KL, với `beta_end = 1.0` và KL annealing trong các epoch đầu.

In [17]:
# Tính VAE loss trên toy example.
x = torch.tensor([[0.50, 1.20, 0.80], [0.10, -0.20, 0.30]])
x_hat = torch.tensor([[0.48, 1.18, 0.75], [0.05, -0.10, 0.28]])
mu = torch.tensor([[0.1, 0.0], [0.2, -0.1]])
logvar = torch.tensor([[0.0, 0.1], [0.2, -0.2]])

recon_loss = torch.mean((x - x_hat) ** 2)
kl_per_sample = -0.5 * torch.sum(1 + logvar - mu.pow(2) - torch.exp(logvar), dim=1)
kl_loss = kl_per_sample.mean()
total_loss = recon_loss + kl_loss

pd.DataFrame(
    [
        {"metric": "recon_loss", "value": float(recon_loss)},
        {"metric": "kl_loss", "value": float(kl_loss)},
        {"metric": "total_loss", "value": float(total_loss)},
    ]
)

,metric,value
0,recon_loss,0.002700
1,kl_loss,0.026326
2,total_loss,0.029026


### 6.7 Batch training

Với batch size thật trong artifact là `1024`:

- Một batch chứa 1024 flow.
- Forward pass sinh 1024 reconstruction.
- Mỗi flow có reconstruction error và KL contribution riêng.
- Batch loss được lấy trung bình.
- Backward pass tính gradient.
- Optimizer cập nhật tất cả weights một chút.

Điểm dễ nhầm: **weights được dùng chung cho mọi flow**. Không có bộ weights riêng cho từng flow.

### 6.8 Gradient và update

Gradient là đạo hàm loss theo weight:

```text
gradient = dLoss/dW
```

Update với gradient descent/Adam về mặt trực giác:

```text
w_new = w_old - learning_rate * gradient
```

Ví dụ gradient dương:

```text
w1,2 = 0.23
gradient = +0.02
lr = 0.001
w_new = 0.23 - 0.001 * 0.02 = 0.22998
```

Ví dụ gradient âm:

```text
w1,2 = 0.23
gradient = -0.02
lr = 0.001
w_new = 0.23 - 0.001 * (-0.02) = 0.23002
```

In [18]:
# Minh họa cập nhật weight với gradient dương và âm.
w_old = 0.23
lr = 0.001

examples = []
for grad in [0.02, -0.02]:
    w_new = w_old - lr * grad
    examples.append({"w_old": w_old, "gradient": grad, "learning_rate": lr, "w_new": w_new})

pd.DataFrame(examples)

,w_old,gradient,learning_rate,w_new
0,0.23,0.02,0.001,0.22998
1,0.23,-0.02,0.001,0.23002


### 6.9 Epoch và best checkpoint

- Một epoch nghĩa là model đã đi qua toàn bộ training batches một lần.
- Weights không reset giữa các epoch.
- Sau mỗi epoch, validation chạy ở chế độ không update weights.
- Epoch có validation loss tốt nhất được lưu thành `vae_best.pth`.
- Best checkpoint không nhất thiết là epoch cuối.

Kết quả cuối cùng dùng trong báo cáo phải lấy từ `artifacts/experiments/best_epoch/best_epoch_summary.csv`:

| Trường | Giá trị |
|---|---:|
| Best Epoch | 58 |
| Final Epoch | 68 |
| Best Train Loss | 0.772250 |
| Best Validation Loss | 0.740856 |
| Final Train Loss | 0.765520 |
| Final Validation Loss | 0.746338 |

`artifacts/models/training_history.json` vẫn được dùng để minh họa learning curve theo cấu trúc thật của artifact hiện có, nhưng không được dùng để thay thế số Best Epoch / Final Epoch trong báo cáo khi nó khác `best_epoch_summary.csv`.


In [19]:
# Xem Best Epoch trong history_df nếu epoch đó có trong training_history.json.
# Với artifact hiện tại, history_df được tạo từ cấu trúc dict các list trong training_history.json.
history_columns = [
    "epoch",
    "train_loss",
    "train_recon",
    "train_kl",
    "val_loss",
    "val_recon",
    "val_kl",
    "beta",
]
available_history_columns = [col for col in history_columns if col in history_df.columns]

best_epoch_history_rows = history_df.loc[
    history_df["epoch"] == best_epoch,
    available_history_columns,
].copy()

if best_epoch_history_rows.empty:
    display(
        pd.DataFrame(
            [
                {
                    "note": "Best Epoch báo cáo không nằm trong training_history.json hiện có.",
                    "best_epoch_from_report_csv": best_epoch,
                    "report_source": BEST_EPOCH_SUMMARY_PATH.relative_to(PROJECT_ROOT).as_posix(),
                }
            ]
        )
    )
else:
    display(best_epoch_history_rows)


,note,best_epoch_from_report_csv,report_source
0,Best Epoch báo cáo không nằm trong training_hi...,58,artifacts/experiments/best_epoch/best_epoch_su...


## 7. Xác định threshold

Threshold runtime hiện tại được đọc trực tiếp từ `artifacts/threshold/threshold.json`.

```text
Validation BENIGN
↓
Reconstruction Error
↓
P99 percentile
↓
Threshold = 6.0312628746032715
```

Quy tắc phân loại:

```text
error <= threshold -> Normal
error > threshold  -> Anomaly
```

Khi bảo vệ, hãy nhấn mạnh rằng threshold là percentile trên reconstruction error của validation BENIGN, và backend production dùng đúng file `threshold.json` này thay vì hard-code giá trị ở nhiều nơi.


In [20]:
# Đọc threshold runtime từ JSON hợp lệ. Đây là nguồn phân loại cuối cùng của backend.
threshold_data = load_json(THRESHOLD_PATH)
threshold_value = float(threshold_data["threshold"])
threshold_percentile = float(threshold_data["percentile"])

threshold_candidates = pd.DataFrame(
    [
        {
            "source": THRESHOLD_PATH.relative_to(PROJECT_ROOT).as_posix(),
            "percentile": threshold_percentile,
            "threshold": threshold_value,
            "note": "Runtime threshold artifact",
        }
    ]
)

threshold_candidates


,source,percentile,threshold,note
0,artifacts/threshold/threshold.json,99.0,6.031263,Runtime threshold artifact


Khi bảo vệ, nên nói rõ:

- Ý tưởng threshold là percentile trên validation BENIGN.
- Runtime hiện tại dùng **P99** và threshold `6.0312628746032715`.
- Backend đọc threshold trực tiếp từ `artifacts/threshold/threshold.json`.
- Nếu threshold thay đổi trong tương lai, phải cập nhật artifact và các tài liệu liên quan cùng nhau.


## 8. Chạy thực tế khi upload CSV

Pipeline inference thật:

```text
User upload CSV
-> backend đọc file
-> validate đủ 66 required columns
-> reorder theo feature_columns.json
-> convert numeric, thay ±Inf bằng NaN
-> fill NaN bên trong cột bằng imputation_medians.json
-> scale bằng scaler.joblib
-> chạy vae_best.pth
-> tính reconstruction error
-> so sánh threshold.json
-> trả JSON
-> dashboard hiển thị kết quả
```

Cell dưới đây ưu tiên gọi `VAEInferenceService` thật. Nếu môi trường notebook không load được service, cell fallback sang pipeline thủ công để minh họa nhưng vẫn dùng model, scaler, schema, median và threshold runtime thật.


In [21]:
def load_model_from_artifacts():
    '''Tải VAE từ backend model class và artifact config/checkpoint.'''
    import torch
    from backend.app.models.vae import VAE

    config = load_json(MODEL_CONFIG_PATH)
    model = VAE(
        input_dim=int(config["input_dim"]),
        latent_dim=int(config["latent_dim"]),
        hidden_dims=list(config["hidden_dims"]),
    )

    try:
        state = torch.load(MODEL_CHECKPOINT_PATH, map_location="cpu", weights_only=True)
    except TypeError:
        state = torch.load(MODEL_CHECKPOINT_PATH, map_location="cpu")

    model.load_state_dict(state)
    model.eval()
    return model


def run_manual_prediction(df: pd.DataFrame, threshold_value: float) -> dict:
    '''Mô phỏng inference không qua FastAPI, chỉ dùng cho notebook demo.'''
    import torch
    from backend.app.models.vae import VAE

    features = load_feature_columns()
    medians_local = load_imputation_medians()
    scaler_local = load_scaler()
    model = load_model_from_artifacts()

    aligned, _extra_cols = validate_and_align_schema(df, features)
    clean = clean_and_impute(aligned, medians_local)
    original_indices = clean.index.tolist()
    X_scaled = scaler_local.transform(clean)

    with torch.no_grad():
        x = torch.tensor(X_scaled, dtype=torch.float32)
        x_hat, _mu, _logvar = model(x)
        errors = VAE.reconstruction_error(x, x_hat).cpu().numpy()

    predictions = (errors > float(threshold_value)).astype(int)
    results = [
        {
            "row_index": int(original_indices[i]),
            "reconstruction_error": float(errors[i]),
            "prediction": int(predictions[i]),
            "prediction_label": "anomaly" if predictions[i] == 1 else "normal",
        }
        for i in range(len(predictions))
    ]

    anomaly_count = int(predictions.sum())
    total = len(predictions)
    return {
        "total_flows": total,
        "anomaly_count": anomaly_count,
        "normal_count": total - anomaly_count,
        "anomaly_rate": anomaly_count / total if total else 0.0,
        "threshold": float(threshold_value),
        "results": results,
    }


try:
    # Đây là pipeline backend thật: preprocess -> scaler -> VAE -> threshold.
    from backend.app.core.inference import VAEInferenceService

    service = VAEInferenceService()
    prediction_result = service.predict_dataframe(sample_df)
    print("Đã chạy VAEInferenceService thật.")
except Exception as exc:
    print("Không chạy được VAEInferenceService, dùng fallback minh họa:")
    print(" ", exc)

    prediction_result = run_manual_prediction(sample_df, threshold_value)

summary = {k: prediction_result[k] for k in ["total_flows", "anomaly_count", "normal_count", "anomaly_rate", "threshold"]}
summary


Đã chạy VAEInferenceService thật.


{'total_flows': 128,
 'anomaly_count': 1,
 'normal_count': 127,
 'anomaly_rate': 0.007812,
 'threshold': 6.0312628746032715}

In [22]:
# Hiển thị 10 flow đầu tiên sau dự đoán.
pd.DataFrame(prediction_result["results"]).head(10)

,row_index,reconstruction_error,prediction,prediction_label
0,0,0.462908,0,normal
1,1,0.301657,0,normal
2,2,0.303204,0,normal
3,3,0.285458,0,normal
4,4,0.462903,0,normal
5,5,0.212796,0,normal
6,6,0.344742,0,normal
7,7,0.197275,0,normal
8,8,0.284574,0,normal
9,9,0.769003,0,normal


In [23]:
# Kiểm tra số anomaly khi dùng threshold runtime hiện tại.
comparison_rows = []
for _, row in threshold_candidates.iterrows():
    result = run_manual_prediction(sample_df, float(row["threshold"]))
    comparison_rows.append(
        {
            "source": row["source"],
            "percentile": row["percentile"],
            "threshold": row["threshold"],
            "anomaly_count": result["anomaly_count"],
            "normal_count": result["normal_count"],
            "note": row["note"],
        }
    )

pd.DataFrame(comparison_rows)


,source,percentile,threshold,anomaly_count,normal_count,note
0,artifacts/threshold/threshold.json,99.0,6.031263,1,127,Runtime threshold artifact


## 9. API demo

Chạy backend từ repo root:

```powershell
uvicorn backend.app.main:app --host 127.0.0.1 --port 8000 --reload
```

Demo bằng `curl`:

```bash
curl -F "file=@artifacts/sample_batch/fixed_batch.csv" http://localhost:8000/predict
```

Demo bằng PowerShell:

```powershell
Invoke-RestMethod -Uri "http://127.0.0.1:8000/predict" -Method POST -Form @{file=Get-Item "artifacts\sample_batch\fixed_batch.csv"}
```

Các field output chính:

| Field | Ý nghĩa |
|---|---|
| `status` | Trạng thái request, thường là `ok` |
| `summary.total_flows` | Tổng số flow đã xử lý |
| `summary.anomaly_count` | Số flow bị phân loại anomaly |
| `summary.normal_count` | Số flow normal |
| `summary.anomaly_rate` | Tỷ lệ anomaly |
| `summary.threshold` | Threshold dùng để phân loại |
| `results[].reconstruction_error` | MSE reconstruction error của từng flow |
| `results[].prediction` | `0 = normal`, `1 = anomaly` |
| `results[].prediction_label` | Nhãn chuỗi `normal` hoặc `anomaly` |

Backend đọc `artifacts/threshold/threshold.json` trực tiếp, nên đây là file cần kiểm tra đầu tiên nếu kết quả phân loại không như mong đợi.


## 10. Kết luận

Những ý chính cần nắm:

- Project dùng cùng preprocessing cho training và inference.
- Feature order 66 cột là contract bắt buộc.
- Missing required columns là lỗi schema.
- Median imputation chỉ xử lý NaN trong các cột đã có, không tạo cột bị thiếu.
- Scaler được fit trên train set và backend chỉ dùng `transform`.
- VAE học hành vi BENIGN.
- Reconstruction error là anomaly score.
- Threshold runtime là P99, giá trị `6.0312628746032715`.
- Best Epoch / Final Epoch báo cáo là `58 / 68`, đọc từ `best_epoch_summary.csv`.

Artifacts dùng khi deploy:

- `artifacts/models/vae_best.pth`
- `artifacts/models/model_config.json`
- `artifacts/scaler/scaler.joblib`
- `artifacts/scaler/imputation_medians.json`
- `artifacts/feature_schema/feature_columns.json`
- `artifacts/threshold/threshold.json`


## 11. Final verification

Bảng dưới đây gom các giá trị phải kiểm tra trước khi demo hoặc bảo vệ notebook. Tất cả đều lấy từ runtime artifacts hiện có, không hard-code kết quả dự đoán.


In [24]:
# Final verification bằng artifact runtime thật và sample CSV thật.
verification_rows = [
    {
        "item": "Model checkpoint path",
        "value": MODEL_CHECKPOINT_PATH.relative_to(PROJECT_ROOT).as_posix(),
    },
    {
        "item": "Threshold path",
        "value": THRESHOLD_PATH.relative_to(PROJECT_ROOT).as_posix(),
    },
    {
        "item": "Threshold value",
        "value": threshold_value,
    },
    {
        "item": "Best Epoch",
        "value": best_epoch,
    },
    {
        "item": "Final Epoch",
        "value": final_epoch,
    },
    {
        "item": "Sample CSV result",
        "value": (
            f"{prediction_result['total_flows']} flows, "
            f"{prediction_result['anomaly_count']} anomaly, "
            f"{prediction_result['normal_count']} normal, "
            f"threshold={prediction_result['threshold']}"
        ),
    },
]

pd.DataFrame(verification_rows)


,item,value
0,Model checkpoint path,artifacts/models/vae_best.pth
1,Threshold path,artifacts/threshold/threshold.json
2,Threshold value,6.031263
3,Best Epoch,58
4,Final Epoch,68
5,Sample CSV result,"128 flows, 1 anomaly, 127 normal, threshold=6...."
